# Proceso Gaussiano — Búsqueda Bayesiana
Modelado y predicción usando `GaussianProcessRegressor` de scikit-learn.

|                |   |
:----------------|---|
| **Nombre**   |  Diego Dueñas Martín   |
| **Fecha**   |  23/03/2026    |
| **Expediente** | 751426|   

## 1. Librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

## 2. Rango y puntos observados
Se define el rango de búsqueda $[2, 15]$ y los pares $(x, y)$ conocidos.

In [ ]:
# Rango de búsqueda
range_min = 2
range_max = 15

# Puntos observados
X_obs = np.array([[10], [2], [3], [7], [12], [9.9651], [9.8741]])
y_obs = np.array([2847, -4017, -1255, 1773, 563, 2849.5981, 2852.69711])

## 3. Definición y entrenamiento del Proceso Gaussiano
El GP aprende la distribución subyacente a partir de los puntos observados.
El parámetro `n_restarts_optimizer=10` reinicia la optimización del kernel 10 veces para evitar mínimos locales.

In [ ]:
gpr = GaussianProcessRegressor(n_restarts_optimizer=10, random_state=42)
gpr.fit(X_obs, y_obs)

print(' Modelo entrenado')
print(f'   Kernel resultante: {gpr.kernel_}')

## 4. Predicción en 10,001 puntos equidistantes
Se genera una grilla densa en $[2, 15]$ y el GP predice la media $\mu$ y la desviación estándar $\sigma$ en cada punto.

In [ ]:
X_pred = np.linspace(range_min, range_max, 10001).reshape(-1, 1)
mu, sigma = gpr.predict(X_pred, return_std=True)
X_flat = X_pred.flatten()

# Primeros 30 resultados
print(f"{'X':>10} {'Media ':>12} {'Desv. ':>12}")
print('-' * 37)
for i, (x, m, s) in enumerate(zip(X_flat, mu, sigma)):
    print(f"{x:>10.4f} {m:>12.2f} {s:>12.4f}")
    if i == 30:
        print('  ...')
        break

## 5. Mínimo y máximo predichos
Se identifican los puntos donde la media $\mu$ alcanza su valor más bajo y más alto.

In [ ]:
idx_min = np.argmin(mu)
idx_max = np.argmax(mu)

print('=' * 45)
print(f'📉 MÍNIMO  →  x = {X_flat[idx_min]:.4f}  |  Y = {mu[idx_min]:.2f}')
print(f'📈 MÁXIMO  →  x = {X_flat[idx_max]:.4f}  |  Y = {mu[idx_max]:.2f}')
print('=' * 45)

## 6. Gráfica
Se visualiza:
- La curva de predicción $\mu(x)$
- La banda de incertidumbre $\mu \pm 2\sigma$ (cubre el 95% de probabilidad)
- Los puntos observados
- Los marcadores de mínimo y máximo

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Banda de incertidumbre ±2σ (95%)
ax.fill_between(X_flat, mu - 2*sigma, mu + 2*sigma,
                alpha=0.2, color='steelblue', label='Incertidumbre ±2 desv.')

# Curva predicha
ax.plot(X_flat, mu, color='steelblue', linewidth=2, label='Predicción GP (Media)')

# Puntos observados
ax.scatter(X_obs, y_obs, color='black', zorder=5, s=50, label='Puntos observados')

# Mínimo y máximo
ax.scatter(X_flat[idx_min], mu[idx_min], color='red', zorder=6, s=150,
           marker='v', label=f'Mínimo: ({X_flat[idx_min]:.2f}, {mu[idx_min]:.2f})')
ax.scatter(X_flat[idx_max], mu[idx_max], color='green', zorder=6, s=150,
           marker='^', label=f'Máximo: ({X_flat[idx_max]:.2f}, {mu[idx_max]:.2f})')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Proceso Gaussiano — Predicción en [2, 15]')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Reflexion  
El metodo de buscade bayesiano se presenta como una forma facil y efectiva, aunque larga para encontrar variables necesarias, como funcionando como una regresio linear. 